In [15]:
import os
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

# Load data
BASE_FILE = os.path.join("..", "..", "data", "processed",
                         "cardio_onc_prostate_06_broad_clean.csv")
OUT_DIR = os.path.join("..", "results", "risk_phenotype")
os.makedirs(OUT_DIR, exist_ok=True)

df = pd.read_csv(BASE_FILE)
print(f"Dataset shape: {df.shape}")

# Define target and variables
target = "at_risk"

# Remove patients with unknown risk status
df_clean = df[df[target].notna()].copy()
print(f"Patients with known risk status: {len(df_clean)}")
print(f"At risk: {(df_clean[target] == 1).sum()}")
print(f"Not at risk: {(df_clean[target] == 0).sum()}")

Dataset shape: (239, 60)
Patients with known risk status: 203
At risk: 79
Not at risk: 124


In [16]:

demographics = ["age", "ethnicity", "bmi"]

risk_factors = [
    "hx_htn",
    "hx_hld",
    "hx_dm2",
    "hx_smoking",
    "hx_cad",
    "hx_chf",
    "hx_arrhythmia",
    "hx_pad",
    "hx_cva"
]

care_patterns = [
    "has_pcp",
    "cards_prior",
    "diet_counseling",
    "exercise_counseling",
    "echo_ordered",
    "ecg_done"
]

print("\n" + "="*80)
print("RISK PHENOTYPE ANALYSIS")
print("="*80)


RISK PHENOTYPE ANALYSIS


In [17]:
continuous_results = []

for col in ["age", "bmi"]:
    not_at_risk = df_clean[df_clean[target] == 0][col].dropna()
    at_risk = df_clean[df_clean[target] == 1][col].dropna()

    # Summary statistics
    nar_mean = not_at_risk.mean()
    nar_std = not_at_risk.std()
    ar_mean = at_risk.mean()
    ar_std = at_risk.std()

    # Test for significance
    _, p = stats.mannwhitneyu(not_at_risk, at_risk, alternative='two-sided')

    # Calculate effect size (Cohen's d)
    pooled_std = np.sqrt(((len(not_at_risk)-1)*nar_std**2 + (len(at_risk)-1)*ar_std**2) /
                         (len(not_at_risk) + len(at_risk) - 2))
    cohens_d = (ar_mean - nar_mean) / pooled_std

    continuous_results.append({
        'Variable': col,
        'Not_At_Risk_Mean': nar_mean,
        'Not_At_Risk_SD': nar_std,
        'At_Risk_Mean': ar_mean,
        'At_Risk_SD': ar_std,
        'Difference': ar_mean - nar_mean,
        'Cohens_d': cohens_d,
        'p_value': p,
        'Significant': 'Yes' if p < 0.05 else 'Trend' if p < 0.10 else 'No'
    })

    print(f"{col.upper()}")
    print(f"  Not at risk: {nar_mean:.2f} ± {nar_std:.2f}")
    print(f"  At risk:     {ar_mean:.2f} ± {ar_std:.2f}")
    print(f"  Difference:  {ar_mean - nar_mean:.2f}")
    print(f"  p-value:     {p:.4f}")
    print(f"  Cohen's d:   {cohens_d:.3f}")
    print()

cont_df = pd.DataFrame(continuous_results)
cont_df.to_csv(os.path.join(OUT_DIR, "continuous_risk_factors.csv"), index=False)


AGE
  Not at risk: 71.77 ± 9.38
  At risk:     71.41 ± 8.46
  Difference:  -0.37
  p-value:     0.8862
  Cohen's d:   -0.041

BMI
  Not at risk: 27.80 ± 5.22
  At risk:     27.52 ± 5.11
  Difference:  -0.28
  p-value:     0.8451
  Cohen's d:   -0.055



In [18]:
categorical_results = []

all_categorical = ["ethnicity"] + risk_factors + care_patterns

for col in all_categorical:
    ct = pd.crosstab(df_clean[col], df_clean[target])

    # Skip if insufficient data
    if ct.shape[0] < 2 or ct.sum().sum() < 10:
        continue

    # Calculate percentages
    pct = ct.div(ct.sum(axis=0), axis=1) * 100

    # Chi-square test
    try:
        chi2, p, dof, expected = stats.chi2_contingency(ct)
    except:
        p = np.nan
        chi2 = np.nan

    # For each category, calculate risk
    for category in ct.index:
        if 0 in ct.columns and 1 in ct.columns:
            count_nar = ct.loc[category, 0]
            count_ar = ct.loc[category, 1]
            pct_nar = pct.loc[category, 0]
            pct_ar = pct.loc[category, 1]

            # Calculate relative risk
            total_ar = ct[1].sum()
            total_nar = ct[0].sum()

            risk_in_category = count_ar / (count_ar + count_nar) if (count_ar + count_nar) > 0 else 0
            overall_risk = total_ar / (total_ar + total_nar)
            relative_risk = risk_in_category / overall_risk if overall_risk > 0 else np.nan

            categorical_results.append({
                'Variable': col,
                'Category': category,
                'Not_At_Risk_N': int(count_nar),
                'Not_At_Risk_Pct': pct_nar,
                'At_Risk_N': int(count_ar),
                'At_Risk_Pct': pct_ar,
                'Risk_in_Category': risk_in_category * 100,
                'Relative_Risk': relative_risk,
                'p_value': p,
                'Significant': 'Yes' if p < 0.05 else 'Trend' if p < 0.10 else 'No'
            })

    print(f"{col.upper()} (p={p:.4f})")
    print(ct)
    print("\nPercentages:")
    print(pct.round(1))
    print()

cat_df = pd.DataFrame(categorical_results)
cat_df.to_csv(os.path.join(OUT_DIR, "categorical_risk_factors.csv"), index=False)


ETHNICITY (p=0.3554)
at_risk    0.0  1.0
ethnicity          
Asian       15   12
Black        8    9
Caucasian   71   39
Hispanic    15    7
Other       11   12

Percentages:
at_risk     0.0   1.0
ethnicity            
Asian      12.5  15.2
Black       6.7  11.4
Caucasian  59.2  49.4
Hispanic   12.5   8.9
Other       9.2  15.2

HX_HTN (p=0.0561)
at_risk  0.0  1.0
hx_htn           
0.0       57   25
1.0       65   53

Percentages:
at_risk   0.0   1.0
hx_htn             
0.0      46.7  32.1
1.0      53.3  67.9

HX_HLD (p=1.0000)
at_risk  0.0  1.0
hx_hld           
0.0       59   37
1.0       65   42

Percentages:
at_risk   0.0   1.0
hx_hld             
0.0      47.6  46.8
1.0      52.4  53.2

HX_DM2 (p=0.7004)
at_risk  0.0  1.0
hx_dm2           
0.0       88   53
1.0       35   25

Percentages:
at_risk   0.0   1.0
hx_dm2             
0.0      71.5  67.9
1.0      28.5  32.1

HX_SMOKING (p=0.0720)
at_risk     0.0  1.0
hx_smoking          
0.0          87   45
1.0          30   31
2.0      

In [19]:
# Identify high-risk characteristics
high_risk_continuous = cont_df[cont_df['p_value'] < 0.10].sort_values('p_value')
high_risk_categorical = cat_df[
    (cat_df['p_value'] < 0.10) & (cat_df['Relative_Risk'] > 1.0)
].sort_values('p_value')

print("HIGH-RISK PROFILE CHARACTERISTICS:\n")
print("-" * 80)

# Continuous variables
if not high_risk_continuous.empty:
    print("\nDEMOGRAPHIC FACTORS:")
    for _, row in high_risk_continuous.iterrows():
        direction = "Higher" if row['Difference'] > 0 else "Lower"
        print(f"\n  • {row['Variable'].upper()}")
        print(f"    - {direction} in at-risk patients: {row['At_Risk_Mean']:.1f} vs {row['Not_At_Risk_Mean']:.1f}")
        print(f"    - Difference: {abs(row['Difference']):.1f}")
        print(f"    - p-value: {row['p_value']:.4f}")
        if row['p_value'] < 0.05:
            print(f"    - ✓ STATISTICALLY SIGNIFICANT")
else:
    print("\nDEMOGRAPHIC FACTORS: No significant differences in age or BMI")

# Categorical variables - group by type
print("\n" + "-" * 80)
print("\nCARDIOVASCULAR COMORBIDITIES:")

cv_comorbidities = high_risk_categorical[
    high_risk_categorical['Variable'].isin(risk_factors)
].sort_values('Relative_Risk', ascending=False)

if not cv_comorbidities.empty:
    for _, row in cv_comorbidities.iterrows():
        print(f"\n  • {row['Variable'].upper()} = {row['Category']}")
        print(f"    - Present in {row['At_Risk_Pct']:.1f}% of at-risk vs {row['Not_At_Risk_Pct']:.1f}% of not-at-risk")
        print(f"    - Relative risk: {row['Relative_Risk']:.2f}x")
        print(f"    - p-value: {row['p_value']:.4f}")
        if row['p_value'] < 0.05:
            print(f"    - ✓ STATISTICALLY SIGNIFICANT")
else:
    print("\n  No single comorbidity significantly associated with increased risk")

print("\n" + "-" * 80)
print("\nCARE PATTERNS:")

care_patterns_risk = high_risk_categorical[
    high_risk_categorical['Variable'].isin(care_patterns)
].sort_values('Relative_Risk', ascending=False)

if not care_patterns_risk.empty:
    for _, row in care_patterns_risk.iterrows():
        print(f"\n  • {row['Variable'].upper()} = {row['Category']}")
        print(f"    - Present in {row['At_Risk_Pct']:.1f}% of at-risk vs {row['Not_At_Risk_Pct']:.1f}% of not-at-risk")
        print(f"    - Relative risk: {row['Relative_Risk']:.2f}x")
        print(f"    - p-value: {row['p_value']:.4f}")
else:
    print("\n  No care patterns significantly associated with increased risk")


HIGH-RISK PROFILE CHARACTERISTICS:

--------------------------------------------------------------------------------

DEMOGRAPHIC FACTORS: No significant differences in age or BMI

--------------------------------------------------------------------------------

CARDIOVASCULAR COMORBIDITIES:

  • HX_SMOKING = 1.0
    - Present in 39.2% of at-risk vs 24.2% of not-at-risk
    - Relative risk: 1.31x
    - p-value: 0.0720

  • HX_HTN = 1.0
    - Present in 67.9% of at-risk vs 53.3% of not-at-risk
    - Relative risk: 1.15x
    - p-value: 0.0561

--------------------------------------------------------------------------------

CARE PATTERNS:

  No care patterns significantly associated with increased risk


In [20]:
# Create risk score based on significant/trending factors
df_clean['risk_score'] = 0

# Add points for each risk factor present
scoring_factors = []

# Hypertension (if p < 0.10)
htn_row = cat_df[(cat_df['Variable'] == 'hx_htn') & (cat_df['Category'] == 1.0)]
if not htn_row.empty and htn_row.iloc[0]['p_value'] < 0.10:
    df_clean['risk_score'] += df_clean['hx_htn'].fillna(0)
    scoring_factors.append({
        'Factor': 'Hypertension',
        'Points': 1,
        'p_value': htn_row.iloc[0]['p_value']
    })

# Smoking (if p < 0.10)
smoking_row = cat_df[(cat_df['Variable'] == 'hx_smoking') & (cat_df['Category'] != 0.0)]
if not smoking_row.empty:
    smoking_p = smoking_row.iloc[0]['p_value']
    if smoking_p < 0.10:
        df_clean['risk_score'] += (df_clean['hx_smoking'] > 0).astype(int)
        scoring_factors.append({
            'Factor': 'Smoking History',
            'Points': 1,
            'p_value': smoking_p
        })

# Other significant comorbidities
for comorbidity in ['hx_cad', 'hx_hld', 'hx_dm2', 'hx_chf', 'hx_arrhythmia']:
    comorb_row = cat_df[(cat_df['Variable'] == comorbidity) & (cat_df['Category'] == 1.0)]
    if not comorb_row.empty and comorb_row.iloc[0]['p_value'] < 0.10:
        df_clean['risk_score'] += df_clean[comorbidity].fillna(0)
        scoring_factors.append({
            'Factor': comorbidity.replace('hx_', '').upper(),
            'Points': 1,
            'p_value': comorb_row.iloc[0]['p_value']
        })

# Age (if significant - add point for age > median in at-risk group)
age_row = cont_df[cont_df['Variable'] == 'age']
if not age_row.empty and age_row.iloc[0]['p_value'] < 0.10:
    age_threshold = df_clean[df_clean[target] == 1]['age'].median()
    df_clean['risk_score'] += (df_clean['age'] > age_threshold).astype(int)
    scoring_factors.append({
        'Factor': f'Age > {age_threshold:.0f}',
        'Points': 1,
        'p_value': age_row.iloc[0]['p_value']
    })

print("RISK SCORING SYSTEM:\n")
if scoring_factors:
    for factor in scoring_factors:
        print(f"  • {factor['Factor']}: +{factor['Points']} point (p={factor['p_value']:.4f})")

    # Analyze risk score distribution
    print(f"\n\nRISK SCORE DISTRIBUTION:\n")
    score_by_risk = df_clean.groupby(target)['risk_score'].describe()
    print(score_by_risk)

    # Create risk categories
    df_clean['risk_category'] = pd.cut(
        df_clean['risk_score'],
        bins=[-np.inf, 0, 1, 2, np.inf],
        labels=['No Risk Factors', 'Low (1)', 'Moderate (2)', 'High (3+)']
    )

    print("\n\nRISK CATEGORY BY ACTUAL OUTCOME:\n")
    risk_cat_table = pd.crosstab(
        df_clean['risk_category'],
        df_clean[target],
        normalize='columns'
    ) * 100
    print(risk_cat_table.round(1))

    # Save scoring system
    scoring_df = pd.DataFrame(scoring_factors)
    scoring_df.to_csv(os.path.join(OUT_DIR, "risk_scoring_system.csv"), index=False)

else:
    print("No factors met statistical threshold (p < 0.10) for inclusion")
    print("  Consider using clinical judgment to define risk factors")


RISK SCORING SYSTEM:

  • Hypertension: +1 point (p=0.0561)
  • Smoking History: +1 point (p=0.0720)


RISK SCORE DISTRIBUTION:

         count      mean       std  min  25%  50%  75%  max
at_risk                                                    
0.0      124.0  0.822581  0.698993  0.0  0.0  1.0  1.0  2.0
1.0       79.0  1.101266  0.671679  0.0  1.0  1.0  2.0  2.0


RISK CATEGORY BY ACTUAL OUTCOME:

at_risk           0.0   1.0
risk_category              
No Risk Factors  34.7  17.7
Low (1)          48.4  54.4
Moderate (2)     16.9  27.8


In [21]:
# Plot 1: Risk score distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot
axes[0].boxplot([
    df_clean[df_clean[target] == 0]['risk_score'].dropna(),
    df_clean[df_clean[target] == 1]['risk_score'].dropna()
], tick_labels=['Not At Risk', 'At Risk'])
axes[0].set_ylabel('Risk Score')
axes[0].set_title('Risk Score Distribution by Outcome')
axes[0].grid(axis='y', alpha=0.3)

# Histogram
for outcome in [0, 1]:
    subset = df_clean[df_clean[target] == outcome]['risk_score']
    label = 'At Risk' if outcome == 1 else 'Not At Risk'
    axes[1].hist(subset, alpha=0.6, label=label, bins=range(0, int(subset.max())+2))
axes[1].set_xlabel('Risk Score')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Risk Score Histogram')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "risk_score_distribution.png"), dpi=300)
print(f"✓ Saved: risk_score_distribution.png")
plt.close()

# Plot 2: Heatmap of risk factors
if not cat_df.empty:
    # Get top risk factors
    top_risk = cat_df[
        (cat_df['Relative_Risk'] > 1.0) & (cat_df['p_value'] < 0.20)
    ].sort_values('Relative_Risk', ascending=False).head(15)

    if not top_risk.empty:
        plt.figure(figsize=(10, 8))

        # Create matrix for heatmap
        plot_data = top_risk.pivot_table(
            index='Variable',
            columns='Category',
            values='Relative_Risk',
            aggfunc='first'
        )

        sns.heatmap(plot_data, annot=True, fmt='.2f', cmap='RdYlGn_r',
                   center=1.0, vmin=0.5, vmax=2.0, cbar_kws={'label': 'Relative Risk'})
        plt.title('Relative Risk of CV Toxicity by Baseline Characteristic')
        plt.xlabel('Category Value')
        plt.ylabel('Risk Factor')
        plt.tight_layout()
        plt.savefig(os.path.join(OUT_DIR, "relative_risk_heatmap.png"), dpi=300)
        print(f"✓ Saved: relative_risk_heatmap.png")
        plt.close()

✓ Saved: risk_score_distribution.png
✓ Saved: relative_risk_heatmap.png
